# Работа с данными бизнеса в ClickHouse

## Задание 1

Для начала продакт-менеджер хочет понять, где сервис пользуется наибольшей популярностью. Выведите топ-20 городов и регионов России по суммарному количеству прочитанных и прослушанных часов любого контента с мобильных устройств. Для каждой из платформ — iOS и Android — добавьте отдельный столбец с длительностью. Результат должен выглядеть так: город, общая длительность прочитанного и прослушанного контента, длительность на iOS, длительность на Android. Значения округлите до целых чисел для лучшей читаемости. Из выдачи также исключите федеральные округа — оставьте только города и области.

In [ ]:
SELECT usage_geo_id_name AS city,
    round(sum(hours)) AS total_hours,
    round(sumIf(hours, usage_platform_ru = 'Букмейт iOS')) AS ios_hours,
    round(sumIf(hours, usage_platform_ru = 'Букмейт Android')) AS android_hours
FROM source_db.audition
WHERE usage_platform_ru IN ('Букмейт iOS', 'Букмейт Android')
AND usage_country_name = 'Россия'
AND usage_geo_id_name NOT LIKE '%федеральный округ'
GROUP BY usage_geo_id_name
ORDER BY total_hours DESC
LIMIT 20

## Задание 2

С активными регионами определились, а какой контент самый популярный? Получите топ-5 книг по суммарному количеству прочитанных и прослушанных часов на мобильных платформах. Также вычислите среднее время чтения и прослушивания в зависимости от типа книги: текст или аудио. Результат должен выглядеть так: название книги, её автор, суммарное время чтения и прослушивания, среднее время чтения текстовой книги, среднее время прослушивания аудиокниги.
В список включайте только те книги, которые используются в обоих форматах. Числовые значения округляйте до двух знаков после точки.

In [ ]:
WITH books_both AS (
    SELECT c.main_content_name,
        c.main_author_name
    FROM source_db.audition AS a
    JOIN source_db.content AS c ON a.main_content_id = c.main_content_id
    WHERE a.usage_platform_ru IN ('Букмейт iOS', 'Букмейт Android')
    GROUP BY c.main_content_name, c.main_author_name
    HAVING count(DISTINCT c.main_content_type) = 2
)
SELECT c.main_content_name AS title,
    c.main_author_name AS author,
    round(sum(a.hours), 2) AS total_hours,
    round(avgIf(a.hours_sessions_long, c.main_content_type = 'Book'), 2) AS avg_text_hours,
    round(avgIf(a.hours_sessions_long, c.main_content_type = 'Audiobook'), 2) AS avg_audio_hours
FROM source_db.audition AS a
JOIN source_db.content AS c ON a.main_content_id = c.main_content_id
WHERE a.usage_platform_ru IN ('Букмейт iOS', 'Букмейт Android')
AND (c.main_content_name, c.main_author_name) IN (SELECT main_content_name, 
                                                      main_author_name 
                                                  FROM books_both
)
GROUP BY c.main_content_name, c.main_author_name
ORDER BY total_hours DESC
LIMIT 5

## Задание 3

Составьте топ-10 авторов по суммарной длительности чтения их книг на всех платформах, включая веб. Для каждого автора добавьте количество уникальных текстовых книг (тип контента 'Book' ) и выведите среднюю длительность прослушивания их аудиокниг только на мобильных устройствах. Исключите авторов, у которых нет аудиокниг.

In [ ]:
SELECT c.main_author_name AS author,
    round(sumIf(a.hours, c.main_content_type = 'Book'), 2) AS total_reading_hours,
    count(DISTINCT if(c.main_content_type = 'Book', a.main_content_id, NULL)) AS unique_text_books,
    round(avgIf(a.hours_sessions_long,
                c.main_content_type = 'Audiobook'
                AND a.usage_platform_ru IN ('Букмейт iOS', 'Букмейт Android')), 2) AS avg_mobile_audio_hours
FROM source_db.audition AS a
JOIN source_db.content AS c ON a.main_content_id = c.main_content_id
GROUP BY c.main_author_name
HAVING countIf(c.main_content_type = 'Audiobook'
               AND a.usage_platform_ru IN ('Букмейт iOS', 'Букмейт Android')) > 0
ORDER BY total_reading_hours DESC
LIMIT 10

## Задание 4

У продакт-менеджера есть предположение, что среди Android-пользователей аудиокниги почти так же популярны, как тексты. А среди iOS-пользователей читателей книг вдвое больше, чем слушателей, если считать по суммарной длительности сессии.
Проверьте предположение менеджера. Для начала выделите три сегмента пользователей:
«Слушатель» — тот, кто преимущественно пользуется аудиокнигами. Прослушивание книг составляет 70% и выше от суммарной длительности сессий.
«Читатель» — преимущественно пользуется текстовыми книгами. Чтение книг — от 70%.
«Оба» — остальные пользователи сервиса.
Исключите пользователей, у которых нет сессий ни с книгами, ни с аудиокнигами, и посчитайте количество пользователей в каждом из сегментов.
На основе полученных данных проверьте предположение менеджера о том, что среди пользователей Android примерно одинаковое количество читателей и слушателей, а на устройствах iOS читателей книг вдвое больше, чем слушателей. Чтобы определить основную платформу пользователя, учитывайте время её использования. Например, если пользователь посещал сервис с двух устройств: два часа на iOS и пять часов на Android, то основной платформой такого пользователя будет Android.

In [ ]:
WITH user_content_stats AS (
    select a.puid,
        sumIf(a.hours_sessions_long, c.main_content_type = 'Audiobook') AS audio_h,
        sumIf(a.hours_sessions_long, c.main_content_type = 'Book') AS text_h,
        sum(a.hours_sessions_long) AS total_h
    FROM source_db.audition AS a
    JOIN source_db.content AS c ON a.main_content_id = c.main_content_id
    WHERE c.main_content_type IN ('Book', 'Audiobook')
    GROUP BY a.puid
    HAVING total_h > 0
),
user_segments AS (
    select puid,
        multiIf(
            audio_h / total_h >= 0.70, 'Слушатель',
            text_h  / total_h >= 0.70, 'Читатель',
            'Оба'
        ) AS segment
    FROM user_content_stats
),
platform_hours AS (
    select puid,
        usage_platform_ru,
        sum(hours_sessions_long) AS hours
    FROM source_db.audition
    GROUP BY puid, usage_platform_ru
),
main_platform AS (
    SELECT puid,
        argMax(usage_platform_ru, hours) AS platform
    FROM platform_hours
    GROUP BY puid
)
SELECT m.platform,
    s.segment,
    count(*) AS users
FROM user_segments AS s
JOIN main_platform AS m ON s.puid = m.puid
WHERE m.platform IN ('Букмейт iOS', 'Букмейт Android')
GROUP BY m.platform, s.segment
ORDER BY m.platform, users DESC;

## Задание 5

Изучите, существует ли связь между форматом использования приложения (прослушивание или чтение) и днём недели. Падает ли использование аудиокниг в выходные на всех платформах, включая веб? Чтобы это узнать, для каждого типа контента посчитайте среднее время его использования в рабочие и выходные дни и округлите до целого числа. Используя оконные функции, вы также можете пользоваться и комбинаторами.

In [ ]:
SELECT c.main_content_type,
    if(toDayOfWeek(a.msk_business_dt_str) IN (6, 7), 'weekend', 'weekday') AS day_type,
    round(avg(a.hours_sessions_long), 2) AS avg_hours
FROM source_db.audition AS a
JOIN source_db.content AS c ON a.main_content_id = c.main_content_id
WHERE c.main_content_type IN ('Book', 'Audiobook')
  AND a.hours_sessions_long > 0
GROUP BY c.main_content_type, day_type
ORDER BY c.main_content_type, day_type

## Задание 6

Продакт-менеджер хочет отслеживать обновления приложений на Android и iOS. У него есть предположение, что больший процент пользователей iOS используют последнюю версию приложения и в целом чаще его обновляют.
Для начала изучите, у какой части пользователей на текущий момент стоят последние версии приложения на каждой из платформ. Для этого посчитайте последнюю активную версию каждого пользователя и сравните её с последней версией у каждой платформы. Для каждой платформы выведите процент пользователей с последней версией приложения и округлите его до двух знаков после точки.

In [ ]:
WITH last_user_version AS (
    SELECT puid,
        usage_platform_ru,
        argMax(app_version, msk_business_dt_str) AS last_version
    FROM source_db.audition
    WHERE usage_platform_ru IN ('Букмейт iOS', 'Букмейт Android')
    AND app_version IS NOT NULL
    GROUP BY puid, usage_platform_ru
),
platform_latest AS (
    SELECT usage_platform_ru,
        max(app_version) AS latest_version
    FROM source_db.audition
    WHERE usage_platform_ru IN ('Букмейт iOS', 'Букмейт Android')
    AND app_version IS NOT NULL
    GROUP BY usage_platform_ru
)
SELECT l.usage_platform_ru,
    round(countIf(l.last_version = p.latest_version) / count() * 100, 2) AS pct_on_latest
FROM last_user_version AS l
JOIN platform_latest AS p ON l.usage_platform_ru = p.usage_platform_ru
GROUP BY l.usage_platform_ru
ORDER BY l.usage_platform_ru

## Задание 7

Теперь продакт-менеджер хочет понять, как часто пользователи обновляют приложение на каждой из платформ. Фактом обновления считайте изменение версии у каждого пользователя. Представьте, что любое изменение возможно только в сторону более новой версии.
Проверьте предположение о том, что пользователи iOS чаще обновляют приложение. Посчитайте метрику update_rate, которая покажет среднюю частоту обновлений на пользователя. Округлите её до двух знаков после точки.

In [ ]:
WITH ordered_versions AS (
    SELECT puid,
        usage_platform_ru,
        app_version,
        msk_business_dt_str,
        lagInFrame(app_version) OVER (
            PARTITION BY puid, usage_platform_ru
            ORDER BY msk_business_dt_str
        ) AS prev_version
    FROM source_db.audition
    WHERE usage_platform_ru IN ('Букмейт iOS', 'Букмейт Android')
    AND app_version IS NOT NULL
),
updates AS (
    SELECT puid,
        usage_platform_ru,
        sum(if(app_version != prev_version AND prev_version IS NOT NULL, 1, 0)) AS updates_cnt
    FROM ordered_versions
    GROUP BY puid, usage_platform_ru
)
SELECT usage_platform_ru,
    round(avg(updates_cnt), 2) AS update_rate
FROM updates
GROUP BY usage_platform_ru
ORDER BY usage_platform_ru

## Задание 8

Новая задача — у коллег есть опасения, что не все книги на тему магии верно размечены с точки зрения категорий.  Считается, что у книги должно быть не больше 3–4 категорий с темами. Необходимо найти все книги на магическую тему, которые при этом не входят в художественную литературу, и проверить, правильно ли они размечены.
Начните с подсчёта книг с тегом «Магия». Выведите количество таких книг в каталоге.

In [ ]:
SELECT count(DISTINCT main_content_id) AS magic_books
FROM source_db.content
WHERE arrayExists(x -> x = 'Магия', published_topic_title_list)

## Задание 9

Найдите все книги со словом «магия» в названии, для которых не проставлен тег «Магия». При этом не учитывайте книги с тегом «Художественная литература». Выведите количество таких книг в каталоге.

In [ ]:
SELECT count(DISTINCT main_content_id) AS missed_magic_books
FROM source_db.content
WHERE lower(main_content_name) LIKE '%магия%'
AND NOT arrayExists(x -> x = 'Магия', published_topic_title_list)
AND NOT arrayExists(x -> x = 'Художественная литература', published_topic_title_list)

## Задание 10

Посчитайте среднее количество категорий у книг с тегом «Магия» и среднее количество категорий у книг в каталоге в целом. Округлите значения до двух знаков после точки. Напомним, что коллегам важно, чтобы у каждой книги было не больше 3–4 категорий. Получится ли не превысить рекомендованного количества?

In [1]:
SELECT round(avg(length(published_topic_title_list)), 2) AS avg_categories_all
FROM source_db.content

SyntaxError: invalid syntax (396832509.py, line 1)

In [ ]:
SELECT round(avg(length(published_topic_title_list)), 2) AS avg_categories_magic
FROM source_db.content
WHERE arrayExists(x -> x = 'Магия', published_topic_title_list)

## Задание 11

Продакт-менеджер выяснил, что в приложении одной из мобильных платформ могла возникнуть проблема — длина пользовательской сессии (поле hours_sessions_long ) записывается некорректно, и это происходит как минимум в одной из стран. Чтобы найти аномалию в данных, используйте такую меру дисперсии как коэффициент вариации. Напомним его формулу: коэффициент определяется как отношение стандартного отклонения к среднему. Чем выше этот показатель, тем более подозрительно с точки зрения анализа распределены данные.
Исследуйте коэффициент по странам и мобильным платформам. В какой стране и на какой платформе видна аномалия в данных? Ограничьте выборку одной страной, в которой коэффициент вариации для одной из платформ будет наибольшим.

In [ ]:
SELECT usage_country_name,
    usage_platform_ru,
    round(stddevPop(hours_sessions_long) / avg(hours_sessions_long), 4) AS cv,
    count() AS sessions
FROM source_db.audition
WHERE usage_platform_ru IN ('Букмейт iOS', 'Букмейт Android')
AND hours_sessions_long > 0
GROUP BY usage_country_name, usage_platform_ru
ORDER BY cv DESC
LIMIT 1